In [14]:
import pandas as pd
import numpy as np
from scipy import stats
import json
from pathlib import Path

base_path1 = "/capstor/store/cscs/swissai/a0142/agents_uq/lcb_llm_tool_agent_gpt_oss_20b/2368761_20260524_153248/" #lcb-hard
base_path2 = "/capstor/store/cscs/swissai/a0142/agents_uq/lcb_llm_tool_agent_gpt_oss_20b/2368762_20260524_153248/" #lcb-medium

results = pd.read_csv(base_path2 + "readable/final_logprob_bayes_quality.csv")
results_traj = pd.read_csv(base_path2 + "readable/generation_trajectory_scores.csv")
tool_success = pd.read_csv(base_path2 + "tool_success_by_instance.csv")   

summary_path = Path(base_path2) / "readable" / "analysis_summary.json"
prior = json.loads(summary_path.read_text())["prior"]["prior_Y1"]

results = results[results["split"] == "test"].copy()
results_traj = results_traj[results_traj["split"] == "test"].copy()

results["tool_success"] = (
    tool_success["tool_no_final_before_verify_pass_success_rate"]
    .fillna(prior)
)

In [15]:
from lm_polygraph.ue_metrics import PredictionRejectionArea
from lm_polygraph.ue_metrics.ue_metric import (
    get_random_scores,
    normalize_metric,
)
from sklearn.metrics import roc_auc_score

prrs = {
        #"PRR": PredictionRejectionArea(), 
        "PRR_05": PredictionRejectionArea(max_rejection=0.5), 
        # "Spearmanr": stats.spearmanr,
        # "AUROC": lambda y_true, y_score: roc_auc_score(y_true, y_score),
        }

In [16]:
df = {}
for prr in prrs:
    df[prr] = {}
    for col in ["llm_perplexity", "llm_log_seq_prob", "tool_success", "bayes_state"]:
        if col == "quality":
            continue
        if prr == "Spearmanr":
            normalized_score = prrs[prr](results[col], results["quality"]).statistic
        elif prr == "AUROC":
            normalized_score = prrs[prr](results["quality"].values, results[col].values)
        else:
            ue_metric = prrs[prr](-results[col].values, results["quality"].values)
            oracle_score = prrs[prr](-results["quality"].values, results["quality"].values)
            random_score = get_random_scores(prrs[prr], results["quality"])
            normalized_score = normalize_metric(ue_metric, oracle_score, random_score)
        df[prr][col] = normalized_score
        print("End-to-end {} for {}: {}".format(prr, col, normalized_score))
df = pd.DataFrame(df)
df.style.background_gradient()

End-to-end PRR_05 for llm_perplexity: 0.15560443444225802
End-to-end PRR_05 for llm_log_seq_prob: 0.7888746576833316
End-to-end PRR_05 for tool_success: 0.8423511580519639
End-to-end PRR_05 for bayes_state: 0.8201517828496456


,PRR_05
llm_perplexity,0.155604
llm_log_seq_prob,0.788875
tool_success,0.842351
bayes_state,0.820152


In [17]:
base_path1 = "/capstor/store/cscs/swissai/a0142/agents_uq/lcb_llm_tool_agent_gpt_oss_20b/2368761_20260524_153248/" #lcb-hard

results = pd.read_csv(base_path1 + "readable/final_logprob_bayes_quality.csv")
results_traj = pd.read_csv(base_path1 + "readable/generation_trajectory_scores.csv")
tool_success = pd.read_csv(base_path1 + "tool_success_by_instance.csv")   

summary_path = Path(base_path1) / "readable" / "analysis_summary.json"
prior = json.loads(summary_path.read_text())["prior"]["prior_Y1"]

results = results[results["split"] == "test"].copy()
results_traj = results_traj[results_traj["split"] == "test"].copy()

results["tool_success"] = (
    tool_success["tool_no_final_before_verify_pass_success_rate"]
    .fillna(prior)
)

In [ ]:
df1 = {}
for prr in prrs:
    df1[prr] = {}
    for col in ["llm_perplexity", "llm_log_seq_prob", "tool_success", "bayes_state"]:
        if col == "quality":
            continue
        if prr == "Spearmanr":
            normalized_score = prrs[prr](results[col], results["quality"]).statistic
        elif prr == "AUROC":
            normalized_score = prrs[prr](results["quality"].values, results[col].values)
        else:
            ue_metric = prrs[prr](-results[col].values, results["quality"].values)
            oracle_score = prrs[prr](-results["quality"].values, results["quality"].values)
            random_score = get_random_scores(prrs[prr], results["quality"])
            normalized_score = normalize_metric(ue_metric, oracle_score, random_score)
        df1[prr][col] = normalized_score
        print("End-to-end {} for {}: {}".format(prr, col, normalized_score))
df1 = pd.DataFrame(df1)
df1.style.background_gradient()

End-to-end PRR_05 for llm_perplexity: 0.5781999118869412
End-to-end PRR_05 for llm_log_seq_prob: 0.8137558279804141
End-to-end PRR_05 for tool_success: 0.7472865687165737
End-to-end PRR_05 for bayes_state: 0.9110574432014549


,PRR_05
llm_perplexity,0.578200
llm_log_seq_prob,0.813756
tool_success,0.747287
bayes_state,0.911057


In [22]:
df_final = pd.concat([df, df1], axis=1, keys=["lcb-medium", "lcb-hard"])
df_final.style.background_gradient()

,lcb-medium,lcb-hard
,PRR_05,PRR_05
llm_perplexity,0.155604,0.578200
llm_log_seq_prob,0.788875,0.813756
tool_success,0.842351,0.747287
bayes_state,0.820152,0.911057


In [29]:
df_final.columns = ["LCB-Medium", "LCB-Hard"]
df_final.index = ["Perplexity", "Seq. Prob.", "Tool Success Rate", "Bayes Belief State"]

In [31]:
df_final["Average"] = df_final.mean(axis=1)
df_final.style.background_gradient()

,LCB-Medium,LCB-Hard,Average
Perplexity,0.155604,0.578200,0.366902
Seq. Prob.,0.788875,0.813756,0.801315
Tool Success Rate,0.842351,0.747287,0.794819
Bayes Belief State,0.820152,0.911057,0.865605


In [33]:
print(df_final.round(3).to_latex())

\begin{tabular}{lrrr}
\toprule
 & LCB-Medium & LCB-Hard & Average \\
\midrule
Perplexity & 0.156000 & 0.578000 & 0.367000 \\
Seq. Prob. & 0.789000 & 0.814000 & 0.801000 \\
Tool Success Rate & 0.842000 & 0.747000 & 0.795000 \\
Bayes Belief State & 0.820000 & 0.911000 & 0.866000 \\
\bottomrule
\end{tabular}

